## THE BIG PICTURE :

* The goal : <b> Predict Houses Prices </b>

<b>Predict Houses prices (values of <code>median_house_value</code> column) using the other columns as features</b> like : 

* location(longitude/latitude)  
* median_income 
* house age (housing_median_age) 
* number of rooms 
* population 
* ocean_proximity

**Simply :**

* we're going to train a model to answer this question : <b>Given these house characteristics , what is the reasonable price for that house ?</b>

* Then, the trained model will act like a real estate agent who estimate a price by looking at a house's location, size, and neighborhood income ... etc


**How we're gonna do that :**

* You give the model a sample of the dataset <b>(80% of the entire dataset , which is the training data)</b> where we know the input data <b>houses features | X</b> and its actual price <b>the correct output | Y</b>

* Then the model learns patterns from the given Training data, ex: "higher income neighboorhoods -> higher prices," "closer to the ocean -> higher prices"

* Once trained, we give the model a new unseen data <b>(the remaining 20% of the dataset | the Testing data)</b> and ask the model to guess the price

* Then compare the model's guesses to the real prices to see how good it is (Evaluation)


## STEP 1: Load the dataset

In [94]:
import pandas as pd
 
df = pd.read_csv("housing.csv")

print(df.head())

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  


## STEP 2: Explore the data

**1- Check dataframe shape (how many rows/columns) :**

In [95]:
print(df.shape) # (20640, 10) , means the dataframe has 20 640 row and 10 columns

(20640, 10)


**2- Check column names and their data types :**

In [96]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB
None


**3- Check for missing values :**

We can use the <code>isna().sum()</code> method :

* <code>df.isna()</code> goes through every cell in the df (dataframe) and marks the cell: <b>True if empty</b>, <b>False if it has a value</b>

* <code>.sum()</code> counts all the cells marked with True (the empty cells) in each column

In [97]:
print(df.isna().sum())

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64


**4/ Look at basic statistics (min, max, mean) for each numeric column :**

<b>What do we calculate?</b>

-> For each numeric column (like median_income, housing_median_age, median_house_value, etc.), we calculate :

- mean : the average value
- min : the smallest value
- max : the largest value
- std (standard deviation) : how spread out the values are (how far away are the values from each other)
- quartiles (25%, 50%, 75%) : values that split the data into four equal parts (50% is the median)

<b> Why do we need this? </b> 

-> Because before you build any model, you need to understand your data. This step answers questions like :

* Does <b>median_house_value</b> range from realistic numbers, or is something broken (like negative prices)?

* Is <b>housing_median_age</b> between 0 and 100 (makes sense) or does it have a value of 9999 (a data error)?

* Are the columns on wildly different scales? (e.g., median_income might be 0-15, while population might be 0-30,000) — this matters a lot for some models later.

In [98]:
print(df.describe())

          longitude      latitude  housing_median_age   total_rooms  \
count  20640.000000  20640.000000        20640.000000  20640.000000   
mean    -119.569704     35.631861           28.639486   2635.763081   
std        2.003532      2.135952           12.585558   2181.615252   
min     -124.350000     32.540000            1.000000      2.000000   
25%     -121.800000     33.930000           18.000000   1447.750000   
50%     -118.490000     34.260000           29.000000   2127.000000   
75%     -118.010000     37.710000           37.000000   3148.000000   
max     -114.310000     41.950000           52.000000  39320.000000   

       total_bedrooms    population    households  median_income  \
count    20433.000000  20640.000000  20640.000000   20640.000000   
mean       537.870553   1425.476744    499.539680       3.870671   
std        421.385070   1132.462122    382.329753       1.899822   
min          1.000000      3.000000      1.000000       0.499900   
25%        296.00000

## Step 3: Clean the data

* Filling in the empty cells of the column <b>total_bedrooms</b> with median values (float values) of the total_bedrooms column cells values instead of "str" values. because in linear regression we don't use any strings

In [99]:
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())
# Let's check
print(df.isna().sum())

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64


**Convert the values of the last column "ocean_proximity" to 0s and 1s , because in Linear regression the values must be of "float" or "int" datatypes not "str"**

In [100]:
df = pd.get_dummies(df, columns=['ocean_proximity']) 

print(df)

       longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0        -122.23     37.88                41.0        880.0           129.0   
1        -122.22     37.86                21.0       7099.0          1106.0   
2        -122.24     37.85                52.0       1467.0           190.0   
3        -122.25     37.85                52.0       1274.0           235.0   
4        -122.25     37.85                52.0       1627.0           280.0   
...          ...       ...                 ...          ...             ...   
20635    -121.09     39.48                25.0       1665.0           374.0   
20636    -121.21     39.49                18.0        697.0           150.0   
20637    -121.22     39.43                17.0       2254.0           485.0   
20638    -121.32     39.43                18.0       1860.0           409.0   
20639    -121.24     39.37                16.0       2785.0           616.0   

       population  households  median_income  media

<b> Explanation of : <code> pd.get_dummies </code> </b> :

Use <code>pd.get_dummies()</code> : it converts a categorical column into multiple new columns, one per category, filled with True/False (or 1/0).

<code>df = pd.get_dummies(df, columns=['ocean_proximity'])</code>

What this does to your data:

Instead of one column ocean_proximity with text values like "INLAND", "NEAR BAY", etc., you get 5 new columns, one per unique category:

* ocean_proximity_<1H OCEAN
* ocean_proximity_INLAND
* ocean_proximity_ISLAND
* ocean_proximity_NEAR BAY
* ocean_proximity_NEAR OCEAN

For each row, only the column matching its actual category gets True (or 1), and the rest are False (or 0).

Example: a row that was "INLAND" becomes:

* ocean_proximity_<1H OCEAN = False
* ocean_proximity_INLAND    = True
* ocean_proximity_ISLAND    = False
* ocean_proximity_NEAR BAY  = False
* ocean_proximity_NEAR OCEAN = False

This is called one-hot encoding , it's the standard way to handle categories that have no natural order (you can't say "INLAND > NEAR BAY", they're just different labels).

## Step 4: Separate features and target

<b>x</b> is the entire dataset without the target column (the input data)

<b>y</b> is the target data (the correct output)

In [101]:
x = df.drop("median_house_value", axis=1)

y = df['median_house_value']

**What does <code>axis=1</code> in <code>x = df.drop("median_house_value", axis=1)</code> ?**

By default : drop() operates on rows (axis=0), not columns. So pandas is trying to find a row labeled "median_house_value" which doesn't exist in the dataset rows, hence you'll get an ERROR!

<b>The Fix :</b>

* 1/ add <code>axis=1</code> , to express to pandas that you're operating on Columns : <code>x = df.drop("median_house_value", axis=1)</code>

  Or equivalently, using the more explicit keyword <b> columns </b> :

* 2/ using the more explicit keyword <b> columns </b> : <code>x = df.drop(<b>columns</b>=["median_house_value"])</code>


## Step 5: Split into train and test sets

In [102]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.25)


**How does the <code>train_test_split()</code> method works ?**

* We give the function the features (the houses features) <b>"x"</b> and the target (correct_prices) <b>"y"</b>

* Then , the method shuffles the rows randomly, then splits them into two groups : 
    - <b>Training group (X_train and y_train)</b>: the bigger chunk (75% if test_size=0.25). This is what the model learns from.

    - <b> Testing group (X_test and y_test) </b> the smaller chunk (25%). This is used only after training, to check how well the model performs on data it never saw during training.

* <code>test_size=0.25</code> means <b>25% of the rows go to testing</b> (20,640 × 0.25 = 5,160 row for testing), <b>75% go to training</b> (20,640 × 0.75 = 15480 row for training)


<b>NOTE</b> : <code>the test_size=0.25</code> is not a fixed rule, it's just a common convention , here is why :

- More test data (ex:  0.3) -> more reliable evaluation (you're testing on more examples), but the model has less data to learn from 

- Less test data (ex : 0.1) -> the model trains on more data (usually better model), but your evaluation is based on fewer examples, so it might not be as trustworthy

So :

* If we have a Big dataset (like this one : 20,640 rows) -> we can afford a smaller test size like 0.1 to 0.2, because even 10% is thousands of rows, still enough to evaluate reliably

* If we have a Small dataset With only a few hundred rows, we might need a bigger test proportion (or better, cross-validation , a more advanced technique) to get a trustworthy evaluation

<b> Conclusion : Test size of 0.2 to 0.25 is a safe, standard default that works fine almost always on most datasets. </b>


## Step 6: Choose and train a model

**What is a Linear regression model :**

* Linear regression tries to find a straight-line that shows the found relationship between your inputs (features) and your output (target).


* For example if we pick only one feature like <code>median_income</code> and plot it on the x-axis with <code>median_house_value</code> on the y-axis : 

    - Linear regression will try to draw <b>the best-fitting straight line</b> based on the found relationship between <code>median_income</code> and <code>median_house_value</code>


In [ ]:
from sklearn.linear_model import LinearRegression

# Create the model
model = LinearRegression()


**What <code>model = LinearRegression()</code> do ?**

* It just <b>creates an empty, untrained model object</b>. Nothing is calculated yet. No formula, no weights, no line, nothing yet

* <b>Create an instance of the Linear Regression model, with nothing learned yet</b>

In [109]:
# Train the model
model.fit(X_train, y_train)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](13,)","[-25833.59,-24357.79, 1045.67,...,147517.12,-31920.71,-22026.4 ]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](13,)","['longitude','latitude','housing_median_age',...,'ocean_proximity_ISLAND', 'ocean_proximity_NEAR BAY','ocean_proximity_NEAR OCEAN']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-2.163e+06
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,13
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(12)


<h3> <b>Model Training Steps :</b> </h3>

**Step 1 : the model receiving the training data :**

<code>model.fit(X_train, y_train)</code>

we're giving the model 2 things at once :

* <code>X_train</code> : <b>all the input features for 75% of the houses </b>(long,lat,age,rooms,bedrooms,population,income,ocean_proximity)

* <code>y_train</code> : <b>the correct answer (actual price)</b> for those exact same houses


**Next steps coming soon ...**

In [107]:
# Evaluate the model and print the R² performance metric. 
print("R² Score:", model.score(X_test, y_test))

R² Score: 0.6575833469092713


**The resulting R² value typically ranges from -∞ to 1 :**

* <b>1.0</b> -> Perfect model, The independent variables explain all variability in the target variable

* <b>0.0</b> -> Baseline model, The model performs no better than a simple flat line predicting the average value of y.

* <b>Negative value</b> -> Poor model, The model fits the data worse than a simple horizontal line of the mean.
